## scalars computation inner product

In [1]:
import torch
import os
import numpy as np
import jax.numpy as jnp
while not os.getcwd().endswith("ScalarEMLP"):
    os.chdir("..")
print(os.getcwd())

c:\Users\ylu174\my_file\research\bilipschitz_scalars_EML\bilipschitz_experiment\ScalarEMLP


In [ ]:
# simulate a number of trajectory initial points
# for instance
torch.manual_seed(26)
x = torch.rand((500, 4, 3))
print(x[0])
n=x.shape[0]
x_arr=jnp.array(x)

tensor([[0.8313, 0.5565, 0.9642],
        [0.4986, 0.6842, 0.1910],
        [0.0405, 0.4311, 0.8639],
        [0.7758, 0.1079, 0.8848]])


comp_inner_products: given a system of trajectory initial points, compute inner products between the four status vectors, $q_1, q_2, p_1, p_2$, firstly we get a matrix with dimension $R^{4 \times 4}$, leading to a $R^{16}$ vector after flattening it

In [ ]:
# square root of one inner product matrix
def compute_mat_sqrt(x,i):
    """
    take the square root of the inner product given the whole dataset x and index i
    """
    U,S,V=torch.linalg.svd(x[i]@x[i].T)
    xi_inner=U@torch.diag(S)@V
    xi_inner_sqrt=U@torch.diag(S).sqrt()@V
    assert torch.allclose(torch.matrix_power(xi_inner_sqrt,2),xi_inner)
    return xi_inner_sqrt.detach().numpy()
compute_mat_sqrt(x,0)


array([[0.93595576, 0.49939024, 0.48132756, 0.7572594 ],
       [0.49939054, 0.6824153 , 0.13751209, 0.13870208],
       [0.48132774, 0.13751219, 0.77705956, 0.2817477 ],
       [0.7572594 , 0.1387021 , 0.28174767, 0.850989  ]], dtype=float32)

In [4]:
from scalaremlp.nn.objax import comp_inner_products

# for each element representing a status of system, for part without square root: compute square root of each vector
print(x[0]@x[0].T)

scalar_no_sqrt = comp_inner_products(x, take_sqrt=False)
print(scalar_no_sqrt[0])
print(scalar_no_sqrt.shape) 


print(torch.allclose(torch.tensor(
    scalar_no_sqrt[0].reshape(4, 4)), x[0]@x[0].T))


tensor([[1.9305, 0.9794, 1.1066, 1.5581],
        [0.9794, 0.7532, 0.4801, 0.6296],
        [1.1066, 0.4801, 0.9338, 0.8423],
        [1.5581, 0.6296, 0.8423, 1.3962]])
[1.9305224  0.97942096 1.1065502  1.5580604  0.97942096 0.75322926
 0.48014483 0.62959826 1.1065502  0.48014483 0.9337895  0.8422622
 1.5580604  0.62959826 0.8422622  1.3962443 ]
(500, 16)
True


In [5]:
sq_li=[compute_mat_sqrt(x,i) for i in range(x.shape[0])]
scalar1=torch.tensor(sq_li).flatten(start_dim=1,end_dim=2)

C:\Users\ylu174\AppData\Local\Temp\ipykernel_5780\182116697.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  scalar1=torch.tensor(sq_li).flatten(start_dim=1,end_dim=2)


In [6]:
scalars = jnp.einsum('bix,bjx->bij', x_arr, x_arr).reshape(n, -1)
sqrt_mat_li_arr = [compute_mat_sqrt(x, i) for i in range(x.shape[0])]  # list of size n, each of them is in size (4, 4)
scalar2=torch.tensor(sqrt_mat_li_arr).flatten(start_dim=1, end_dim=2) # size(n,16)
assert(torch.allclose(scalar1,scalar2))

if take_sqrt=True, we will also consider $L_2$ norm of each vector and insert at the top, so that we end up with a vector with dimension $R^{20}$,

In [7]:
# take square root of each vector norm and insert it at top, resulting in a 5x4 matrix
scalar_sqrt = comp_inner_products(x, take_sqrt=True)
print(scalar_sqrt.shape)
# print(scalar_sqrt)
print(scalar_sqrt[0])
print(torch.tensor(scalar_sqrt[0].reshape(5, 4)))
# print((torch.diag(x[0]@x[0].T).sqrt().reshape(-1, 4), x[0]@x[0].T))
# print(torch.concatenate(
#     (torch.diag(x[0]@x[0].T).sqrt().reshape(-1, 4), x[0]@x[0].T)))
print(torch.allclose(torch.tensor(
    scalar_sqrt[0].reshape(5, 4)), torch.concatenate((torch.diag(x[0]@x[0].T).sqrt().reshape(-1, 4), x[0]@x[0].T))))



(500, 20)
[1.3894324  0.8678878  0.96632785 1.1816279  1.9305224  0.97942096
 1.1065502  1.5580604  0.97942096 0.75322926 0.48014483 0.62959826
 1.1065502  0.48014483 0.9337895  0.8422622  1.5580604  0.62959826
 0.8422622  1.3962443 ]
tensor([[1.3894, 0.8679, 0.9663, 1.1816],
        [1.9305, 0.9794, 1.1066, 1.5581],
        [0.9794, 0.7532, 0.4801, 0.6296],
        [1.1066, 0.4801, 0.9338, 0.8423],
        [1.5581, 0.6296, 0.8423, 1.3962]])
True


comp_scalars: In addition to the previous 20 scalars, add the inner product between status vectors and $\{g, q_1-q_2\}$, $L_2$ norm and $L_2$ norm square of $q_1-q_2$, finally we get R^{30} vector

In [8]:
xx = torch.tensor(scalar_sqrt)
g = torch.tensor([0, 0, -1],dtype=torch.float32)
xg = torch.inner(x, g)
y = x[:, 0, :] - x[:, 1, :]
# print(y*y)
yy = torch.sum(y*y, dim=-1, keepdim=True)
yy = torch.concatenate((yy, yy.sqrt()), dim=-1)
yx = torch.tensor(np.einsum("bx,bix->bi", y, x))
out2 = torch.concatenate((xx, xg, yy, yx), dim=-1)
print(out2.shape)

torch.Size([500, 30])


In [9]:
# the results of comp_scalars_jax and compute_scalars are equivalent
from scalaremlp.nn.objax import compute_scalars_jax,compute_scalars

import jax.numpy as jnp
scalar_sqrt_orig = compute_scalars(x)
scalar_sqrt_jax = compute_scalars_jax(jnp.array(x))
# print(scalar_sqrt_orig.shape)
# print(scalar_sqrt_jax.shape)
print(torch.allclose(torch.tensor(scalar_sqrt_orig, dtype=torch.double),
      torch.tensor(scalar_sqrt_jax, dtype=torch.double)))

True


## MLP setup

## basic MLP
input: R^{30}, derived from scalars computation of the initial vectors
output: a scalar

In [10]:
## basic MLP
from scalaremlp.nn.objax import BasicMLP_objax, InvarianceLayer_objax, EquivarianceLayer_objax

basic_mlp = BasicMLP_objax(n_in=30, n_out=1)
x_in=x[0].unsqueeze(0) # size (1,4,3)

x_in_scalar=compute_scalars(x_in).squeeze(0) # size (30,)

# print(x_in_scalar)
# print(x_in_scalar)
# print(basic_mlp.vars())

# for name, var in basic_mlp.vars().items():
#     print(name, var.value)

basic_mlp(x_in_scalar)

Array([-0.4229442], dtype=float32)

## invariance MLP
input: R^{30}, derived from scalars computation of the initial vectors
output: a scalar

In [11]:
## invariant MLP
invariant_mlp = InvarianceLayer_objax(n_hidden=100,n_layers=3)
# print(x_in.shape)

x_in_arr=jnp.array(x_in) # size (1,4,3)
scalars = compute_scalars_jax(x_in_arr, invariant_mlp.g)
out = invariant_mlp.mlp(scalars)

print(jnp.array(out).sum())

# print(invariant_mlp.H(x_in_arr))
# invariant_mlp(x_in_arr)

print(invariant_mlp(x_in_arr))

0.0063488446
0.0063488446


In [ ]:
## equivariant MLP
from scalaremlp.nn.objax import radial_basis_transform
mu,gamma=radial_basis_transform(x_in)
# print(mu, gamma)
equivariant_mlp = EquivarianceLayer_objax(n_hidden=100,n_layers=3,mu=mu,gamma=gamma)
equivariant_mlp(x_in_arr,t=1)

Array([[-0.73794043, -0.34157872, -2.0892339 , -0.60373974, -0.07456876,
        -1.6440215 ,  0.71762794,  0.13496147,  1.7755133 , -0.86794853,
        -1.0072129 , -1.0376049 ]], dtype=float32)

## batch

In [13]:
B1=x[0,:,:]
B2=x[1,:,:]
M1=B1@B1.T
M2=B2@B2.T
print(M1)
print(M2)

B=torch.stack((M1,M2))


tensor([[1.9305, 0.9794, 1.1066, 1.5581],
        [0.9794, 0.7532, 0.4801, 0.6296],
        [1.1066, 0.4801, 0.9338, 0.8423],
        [1.5581, 0.6296, 0.8423, 1.3962]])
tensor([[1.0277, 0.6632, 0.9276, 0.5914],
        [0.6632, 1.2649, 0.5863, 0.8561],
        [0.9276, 0.5863, 0.8404, 0.4838],
        [0.5914, 0.8561, 0.4838, 1.2409]])


In [14]:
G = torch.diag(-torch.ones(4))
G[0,0] = 1
print(G.unsqueeze(0))
print(x)
G = torch.einsum('bix,bxj->bij', x, G.unsqueeze(0))
# G = torch.einsum('cix,cxj->cij', x, G.unsqueeze(0))
print(G)

print()
scalars = torch.einsum('bij,bkj->bik', G, x)
print(scalars)

# simplified version: since the matrix is symmetric, take the upper trianglar part, flatten it
scalars = torch.triu(scalars).view(-1, 3**2)
# print(scalars)
# print(torch.nonzero(scalars[0]))
scalars = scalars[:, torch.nonzero(scalars[0]).squeeze(-1)]
print(scalars)

tensor([[[ 1.,  0.,  0.,  0.],
         [ 0., -1.,  0.,  0.],
         [ 0.,  0., -1.,  0.],
         [ 0.,  0.,  0., -1.]]])
tensor([[[0.8313, 0.5565, 0.9642],
         [0.4986, 0.6842, 0.1910],
         [0.0405, 0.4311, 0.8639],
         [0.7758, 0.1079, 0.8848]],

        [[0.9933, 0.2009, 0.0252],
         [0.6007, 0.2152, 0.9262],
         [0.9077, 0.1273, 0.0148],
         [0.3939, 0.9395, 0.4506]],

        [[0.3880, 0.9025, 0.8006],
         [0.4978, 0.0414, 0.6510],
         [0.1410, 0.9330, 0.3758],
         [0.9361, 0.7829, 0.1488]],

        ...,

        [[0.0252, 0.2452, 0.6590],
         [0.0977, 0.5769, 0.8378],
         [0.4738, 0.2634, 0.3167],
         [0.9493, 0.1498, 0.1420]],

        [[0.0398, 0.3179, 0.4091],
         [0.1646, 0.4866, 0.9617],
         [0.6538, 0.9140, 0.7065],
         [0.1207, 0.4813, 0.1381]],

        [[0.8252, 0.4266, 0.3060],
         [0.5122, 0.2552, 0.7821],
         [0.4370, 0.0656, 0.9472],
         [0.6435, 0.4729, 0.0290]]])


RuntimeError: einsum(): subscript x has size 4 for operand 1 which does not broadcast with previously seen size 3

In [ ]:
G = torch.diag(-torch.ones(4))
G[0,0] = 1

M1=x[0,:,:]@G@x[0,:,:].T
M2=x[1,:,:]@G@x[1,:,:].T

print(torch.stack((M1,M2)))

tensor([[[-1.2034, -0.8207, -1.4278],
         [-0.8207, -0.7841, -0.9546],
         [-1.4278, -0.9546, -1.2666]],

        [[-0.4329, -0.3149, -0.0702],
         [-0.3149, -1.4828, -0.3064],
         [-0.0702, -0.3064, -0.0881]]])


In [ ]:
N=x.shape[0]
scalars = torch.einsum('bik,bjl->bijkl', x, x) #[N, n, n, dim, dim]
print(scalars.shape)
print(scalars)

torch.Size([2, 3, 3, 4, 4])
tensor([[[[[4.7946e-01, 6.5345e-01, 1.5515e-01, 5.9648e-01],
           [6.5345e-01, 8.9058e-01, 2.1145e-01, 8.1295e-01],
           [1.5515e-01, 2.1145e-01, 5.0204e-02, 1.9302e-01],
           [5.9648e-01, 8.1295e-01, 1.9302e-01, 7.4208e-01]],

          [[3.0151e-01, 3.1612e-01, 4.0512e-01, 4.5032e-01],
           [4.1092e-01, 4.3084e-01, 5.5214e-01, 6.1373e-01],
           [9.7564e-02, 1.0229e-01, 1.3109e-01, 1.4572e-01],
           [3.7510e-01, 3.9328e-01, 5.0400e-01, 5.6023e-01]],

          [[2.2682e-02, 6.1566e-01, 1.6876e-01, 4.4752e-01],
           [3.0913e-02, 8.3908e-01, 2.3001e-01, 6.0992e-01],
           [7.3395e-03, 1.9922e-01, 5.4610e-02, 1.4481e-01],
           [2.8218e-02, 7.6593e-01, 2.0996e-01, 5.5675e-01]]],


         [[[3.0151e-01, 4.1092e-01, 9.7564e-02, 3.7510e-01],
           [3.1612e-01, 4.3084e-01, 1.0229e-01, 3.9328e-01],
           [4.0512e-01, 5.5214e-01, 1.3109e-01, 5.0400e-01],
           [4.5032e-01, 6.1373e-01, 1.4572e-01, 5